# RoadVanta AI — RDD2022 YOLOv8 Automated Cloud Training Pipeline

This Google Colab notebook trains a custom **YOLOv8** model on your road damage dataset (`archive (1).zip` or any RDD2022 dataset) using **GPU Acceleration (Tesla T4 / A100)**.

### Supported Defect Classes in `RDD_SPLIT`:
- `0`: **Longitudinal Crack**
- `1`: **Transverse Crack**
- `2`: **Alligator Crack**
- `3`: **Pothole** (Primary Hazard)
- `4`: **Road Defect Cluster / Patch**

---

### Step 1: Verify Free GPU Acceleration
Go to **Runtime** &rarr; **Change runtime type** &rarr; Select **T4 GPU** (or A100 if Colab Pro).

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Training on GPU: {torch.cuda.get_device_name(0)}")

### Step 2: Install Ultralytics YOLOv8

In [ ]:
!pip install -q ultralytics pyyaml

### Step 3: Mount Google Drive OR Upload `archive (1).zip`
Run either option below depending on whether your zip file is in Google Drive or uploaded directly.

In [ ]:
# Option A: Mount Google Drive (Recommended for large 10GB archive)
from google.colab import drive
drive.mount('/content/drive')

# Set path to where you uploaded archive (1).zip in your Drive:
ZIP_PATH = '/content/drive/MyDrive/archive (1).zip'

# If uploaded directly to Colab files, use:
import os
if not os.path.exists(ZIP_PATH):
    ZIP_PATH = '/content/archive (1).zip'

print(f"Target archive: {ZIP_PATH} (Exists: {os.path.exists(ZIP_PATH)})")

### Step 4: Extract Dataset & Configure `dataset.yaml`

In [ ]:
import os, zipfile

# Unzip fast to local Colab NVMe SSD storage
!mkdir -p /content/dataset
print("Extracting archive... (this takes ~1-2 minutes on Colab SSD)")
!unzip -q "$ZIP_PATH" -d /content/dataset

# Write YOLOv8 configuration
yaml_content = '''
path: /content/dataset/RDD_SPLIT
train: train/images
val: val/images

names:
  0: longitudinal_crack
  1: transverse_crack
  2: alligator_crack
  3: pothole
  4: road_defect_cluster
'''

with open('/content/dataset/rdd2022.yaml', 'w') as f:
    f.write(yaml_content.strip())

print("YAML dataset created at /content/dataset/rdd2022.yaml")

### Step 5: Start High-Speed GPU Training

In [ ]:
from ultralytics import YOLO

# Load baseline YOLOv8 small or nano
model = YOLO('yolov8s.pt')

# Train with GPU acceleration
results = model.train(
    data='/content/dataset/rdd2022.yaml',
    epochs=50,
    imgsz=640,
    batch=32,
    name='roadvanta_yolov8_trained',
    patience=10,
    save=True,
    device=0
)

### Step 6: Validate Detection Accuracy (mAP50 & Precision)

In [ ]:
metrics = model.val()
print(f"mAP50:     {metrics.box.map50:.4f}")
print(f"mAP50-95:  {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall:    {metrics.box.mr:.4f}")

### Step 7: Download `best.pt` to Put into RoadVanta
Once downloaded, simply place `best.pt` into `prism_backend/models/best.pt` in your project folder!

In [ ]:
from google.colab import files
import shutil

weights_path = 'runs/detect/roadvanta_yolov8_trained/weights/best.pt'
print(f"Downloading {weights_path} to your computer...")
files.download(weights_path)

# Also save a copy back to Google Drive for safety
if os.path.exists('/content/drive/MyDrive'):
    shutil.copy(weights_path, '/content/drive/MyDrive/roadvanta_best.pt')
    print("Saved copy to Google Drive: MyDrive/roadvanta_best.pt")